# Kaggle submission — Akkadian → English (ByT5)

Минимальный ноутбук **только для сабмита** в code-competition: Kaggle приватно
перезапускает выбранную версию со скрытым тестом и забирает `submission.csv` из Output.

Здесь нет обучения/оценки — только инференс: грузим обученную модель с HF Hub,
переводим `test.csv` из подключённого датасета соревнования, пишем
`/kaggle/working/submission.csv` (колонки `id,translation`).

**Перед сабмитом:**
1. Справа **Input → + Add Input** → Competitions → **Deep Past Initiative: Machine Translation**.
2. **Internet → On**, **Add-ons → Secrets** → `HF_TOKEN` (Attach).
3. Save Version (Save & Run All) → дождись Output `submission.csv` → **Submit**.

`RUN_NAME` указывает, какую обученную модель брать с Hub (по имени run из конфига).

In [ ]:
RUN_NAME = "byt5-baseline-docs-raw"  # <- какую модель сабмитим (run_name из конфига)
# NORMALIZE должен совпадать с обучением модели:
#   baseline (train_source=src_raw)         -> False
#   exp1+    (train_source=both/нормализ.)   -> True
NORMALIZE = False
NUM_BEAMS = 4
BRANCH = "ml-dev"
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!rm -rf /kaggle/working/repo
!git clone --branch {BRANCH} https://github.com/ObjoradDdd/ml-hits-3-lab.git /kaggle/working/repo
%cd /kaggle/working/repo/ml
!pip install -q -e . sacrebleu

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login, whoami
login(os.environ["HF_TOKEN"])
HUB_ID = f"{whoami()['name']}/akkadian-{RUN_NAME}"
print("модель:", HUB_ID)

In [ ]:
# инференс на тесте соревнования (при приватном перезапуске тут будет скрытый тест)
from akkadian_nmt.decode import predict_kaggle
TEST = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
predict_kaggle(HUB_ID, TEST, out_csv="/kaggle/working/submission.csv",
               num_beams=NUM_BEAMS, candidates_per_model=NUM_BEAMS,
               normalize=NORMALIZE)
import pandas as pd
print(pd.read_csv("/kaggle/working/submission.csv").head())